# Sorting

Notes and runnable examples on sorting — anchored by **Timsort**, the adaptive hybrid that powers Python's `sorted()` and `list.sort()`. We'll see why O(n log n) is the floor for comparison sorts, what makes Timsort special (and so fast on real data), how `key=` works under the hood, and how the textbook sorts stack up against it.

**Contents**

1. **The comparison-sort floor** — why O(n log n)
2. **Timsort** — what Python actually does
3. **`key=`** — decorate-sort-undecorate
4. **The classic sorts**, for contrast
5. **When to use what**
6. **Complexity cheat-sheet**

## 1. The comparison-sort floor — why O(n log n)

Any sort that works only by **comparing pairs of elements** has a hard lower bound of **O(n log n)**. The argument is information-theoretic: there are `n!` possible orderings of the input, each comparison yields one bit (`≤` or `>`), and you need enough bits to single out the right ordering among all `n!` — so at least `log₂(n!) ≈ n log₂ n` comparisons. No comparison sort beats that in the worst case; the best you can do is hit it with a small constant and **adapt** to data that's already partly ordered.

(Sorts that *don't* compare — counting sort, radix sort — can reach O(n) for integers in a bounded range by using values directly as array indices. They're special-purpose; Python's general `sorted` is a comparison sort.)

In [1]:
import math

print(f"{'n':>9}{'log2(n!)':>14}{'n*log2(n)':>14}")
for n in (10, 1000, 1_000_000):
    log2_factorial = math.lgamma(n + 1) / math.log(2)   # lgamma avoids building a huge factorial
    print(f"{n:>9}{log2_factorial:>14.0f}{n*math.log2(n):>14.0f}")


        n      log2(n!)     n*log2(n)
       10            22            33
     1000          8529          9966
  1000000      18488885      19931569


## 2. Timsort — what Python actually does

`list.sort()` and `sorted()` use **Timsort**, written by Tim Peters for CPython in 2002 (and since adopted by Java, V8, Android, Rust, and Swift). It's a **stable, adaptive** hybrid of merge sort and insertion sort, built on one observation: **real data is rarely random** — it arrives in already-ordered streaks. Timsort finds and exploits them:

- It scans for **runs** — maximal ascending (or strictly descending, which it flips in place) stretches — instead of assuming chaos.
- Short runs are extended to a minimum length (**minrun**, ~32–64) using **binary insertion sort**, which is excellent on tiny arrays.
- Runs are then **merged** like merge sort, with **galloping mode**: when one run keeps winning, it binary-searches ahead instead of stepping one element at a time.

The measurable payoffs: **O(n) on already-sorted (or reverse-sorted) data**, O(n log n) worst case, and **stable** (equal elements keep their original order).

In [2]:
import timeit, random
random.seed(0)

N = 1_000_000
rand     = [random.random() for _ in range(N)]
ordered  = sorted(rand)
reversed_ = ordered[::-1]

for label, data in [("random", rand), ("already sorted", ordered), ("reverse sorted", reversed_)]:
    t = timeit.timeit(lambda: sorted(data), number=3)
    print(f"  {label:15}: {t*1000:7.1f} ms")


  random         :   397.0 ms
  already sorted :    58.0 ms
  reverse sorted :    63.1 ms


Already-sorted data is several times faster — Timsort spots the single giant run and runs essentially linearly. Reverse-sorted is nearly as quick: it detects the descending run and flips it in O(n). Only random data pays the full n log n. Now the stability guarantee:

In [3]:
records = [("apple", 2), ("banana", 1), ("cherry", 2), ("date", 1), ("elder", 1)]
by_number = sorted(records, key=lambda r: r[1])
print("sorted by the number:", by_number)
print("ties keep input order -> banana, date, elder (all 1) stay in their original sequence")


sorted by the number: [('banana', 1), ('date', 1), ('elder', 1), ('apple', 2), ('cherry', 2)]
ties keep input order -> banana, date, elder (all 1) stay in their original sequence


## 3. `key=` — decorate-sort-undecorate

You sort by a derived value with `key=`, a function applied to each element. The detail that matters: Python calls `key` **exactly once per element** (n times), caches the results, sorts *those*, then reorders the originals to match. This is the **decorate–sort–undecorate (DSU)** pattern, done in C. So an expensive key is fine — it runs n times, never the O(n log n) you'd get from calling it inside every comparison.

In [4]:
import random
from operator import itemgetter

calls = 0
def slow_key(x):
    global calls
    calls += 1
    return x % 7                 # pretend this is expensive

data = list(range(1000)); random.shuffle(data)
sorted(data, key=slow_key)
print(f"key called {calls} times for {len(data)} elements  (once each, not n log n)")

# common key idioms:
people = [("Ada", 36), ("Alan", 41), ("Grace", 36)]
print("by age, then name:", sorted(people, key=itemgetter(1, 0)))   # tuple key; stable on ties
print("descending       :", sorted([3, 1, 2], reverse=True))


key called 1000 times for 1000 elements  (once each, not n log n)
by age, then name: [('Ada', 36), ('Grace', 36), ('Alan', 41)]
descending       : [3, 2, 1]


## 4. The classic sorts, for contrast

Worth writing once to see what Timsort improves on. **Insertion sort** is O(n²) but is the right tool for tiny arrays (Timsort uses it internally). **Merge sort** is O(n log n) and stable but needs O(n) scratch space. **Quicksort** is O(n log n) average and in-place, but O(n²) worst case and not stable. Raced against the C Timsort, the pure-Python versions are far slower — a reminder that the implementation's **constant factor** matters as much as the asymptotics.

In [5]:
import timeit, random

def insertion_sort(a):
    a = a[:]
    for i in range(1, len(a)):
        k, j = a[i], i - 1
        while j >= 0 and a[j] > k:
            a[j + 1] = a[j]; j -= 1
        a[j + 1] = k
    return a

def merge_sort(a):
    if len(a) <= 1:
        return a
    mid = len(a) // 2
    left, right = merge_sort(a[:mid]), merge_sort(a[mid:])
    out, i, j = [], 0, 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:          # <= keeps it stable
            out.append(left[i]); i += 1
        else:
            out.append(right[j]); j += 1
    out.extend(left[i:]); out.extend(right[j:])
    return out

def quicksort(a):
    if len(a) <= 1:
        return a
    pivot = a[len(a) // 2]
    return (quicksort([x for x in a if x < pivot])
            + [x for x in a if x == pivot]
            + quicksort([x for x in a if x > pivot]))

random.seed(0)
data = [random.random() for _ in range(10_000)]
expected = sorted(data)

for fn, name in [(insertion_sort, "insertion  O(n^2)"),
                 (merge_sort,     "merge      O(n log n)"),
                 (quicksort,      "quick      O(n log n) avg"),
                 (sorted,         "Timsort    (C builtin)")]:
    t = timeit.timeit(lambda: fn(data), number=1)
    print(f"  {name:26}{t*1000:8.2f} ms   correct={fn(data) == expected}")


  insertion  O(n^2)           574.21 ms   correct=True
  merge      O(n log n)         6.91 ms   correct=True
  quick      O(n log n) avg     6.16 ms   correct=True
  Timsort    (C builtin)        0.70 ms   correct=True


## 5. When to use what

| You need... | Reach for | Why |
|---|---|---|
| Sort a list in place | `list.sort()` | Timsort, no copy; mutates the list |
| A sorted copy of any iterable | `sorted(iterable)` | Timsort into a new list |
| Sort by a derived field | `sorted(..., key=...)` | `key` called once per element (DSU) |
| The k smallest / largest, k ≪ n | `heapq.nsmallest` / `nlargest` | O(n log k), skips a full sort |
| Keep a list sorted as you insert | `bisect.insort` | O(log n) to place, O(n) to shift |
| Sort bounded-range integers, fastest | counting / radix sort | O(n), bypasses the comparison floor |

## 6. Complexity cheat-sheet

| Algorithm | Best | Average | Worst | Space | Stable | In-place |
|---|:---:|:---:|:---:|:---:|:---:|:---:|
| **Timsort** (Python) | O(n) | O(n log n) | O(n log n) | O(n) | ✅ | ✗‡ |
| Insertion sort | O(n) | O(n²) | O(n²) | O(1) | ✅ | ✅ |
| Merge sort | O(n log n) | O(n log n) | O(n log n) | O(n) | ✅ | ✗ |
| Quicksort | O(n log n) | O(n log n) | O(n²) | O(log n) | ✗ | ✅ |
| Heapsort | O(n log n) | O(n log n) | O(n log n) | O(1) | ✗ | ✅ |
| Counting sort | O(n + k) | O(n + k) | O(n + k) | O(n + k) | ✅ | ✗ |

<sub>‡ `list.sort()` reorders in place, but Timsort's merge step uses up to O(n) temporary space, so it isn't constant-space. `sorted()` returns a new list. Heapsort is the heaps notebook's "drain the heap." k = the value range (counting sort).</sub>